# 01 · Data exploration

Sanity-check a crop-disease dataset **before** training: does it load, are the
classes balanced, are the images actually leaf photos, and is anything
corrupt or duplicated.

Point `DATA_ROOT` at a dataset folder (or set `NOMA_DATA_DIR`). See
`notebooks/data/README.md` for the expected layout.

In [ ]:
import sys
from pathlib import Path

def _notebook_dir() -> Path:
    """Locate notebooks/ whether Jupyter starts at the repo root, notebooks/, or deeper."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks" / "utils.py").exists():
            return candidate / "notebooks"
        if (candidate / "utils.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the repo (notebooks/utils.py not found)")


NOTEBOOK_DIR = _notebook_dir()
sys.path.insert(0, str(NOTEBOOK_DIR))

import utils

DATA_ROOT = utils.data_dir()  # or utils.data_dir("data/maize_v1")
print("notebook dir:", NOTEBOOK_DIR)
print("data root   :", DATA_ROOT)

In [ ]:
summary = utils.summarize_dataset(DATA_ROOT)
if not summary:
    raise SystemExit(
        f"No train/valid folders found under {DATA_ROOT}.\n"
            "Copy a dataset into notebooks/data/ or set NOMA_DATA_DIR."
    )

for split, counts in summary.items():
    total = sum(counts.values())
    print(f"{split:>6}: {total:>5} images across {len(counts)} classes")
    for name, count in counts.items():
        print(f"          {name:<32} {count:>5}")

In [ ]:
import matplotlib.pyplot as plt

train_counts = summary.get("train", {})
if train_counts:
    names = list(train_counts)
    values = [train_counts[n] for n in names]
    mean = sum(values) / len(values)

    fig, ax = plt.subplots(figsize=(10, 0.4 * len(names) + 2))
    ax.barh(names, values, color="#4c9a2a")
    ax.axvline(mean, color="#c0392b", linestyle="--", label=f"mean = {mean:.0f}")
    ax.set_xlabel("images")
    ax.set_title("Training class balance")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    worst = min(values)
    if mean / worst > 2:
        print(
            f"⚠️  Imbalance detected: largest class is {max(values) / worst:.1f}x the smallest.\n"
            "   Use class_weight / balanced sampling before trusting accuracy."
        )

In [ ]:
train_dir = utils.find_split_dir(DATA_ROOT, "train")
pairs = utils.list_images(train_dir)
paths = [p for p, _ in pairs]

stats = utils.image_stats(paths)
print("sampled files :", stats["count"])
print("colour modes  :", stats["modes"])
print("top raw sizes :")
for size, count in stats["sizes"].items():
    print(f"   {size[0]}x{size[1]:<5} {count:>5} file(s)")

if any(mode not in ("RGB", "RGBA", "L", "P") for mode in stats["modes"]):
    print("⚠️  Unusual colour modes — all are converted to RGB at load time, but worth a look.")

In [ ]:
import random
from PIL import Image

SAMPLES_PER_CLASS = 4
classes = sorted({label for _, label in pairs})

fig, axes = plt.subplots(
    len(classes), SAMPLES_PER_CLASS, figsize=(2.6 * SAMPLES_PER_CLASS, 2.6 * len(classes))
)
axes = axes.reshape(len(classes), SAMPLES_PER_CLASS)

rng = random.Random(0)
for row, class_name in enumerate(classes):
    class_paths = [p for p, label in pairs if label == class_name]
    for col in range(SAMPLES_PER_CLASS):
        ax = axes[row, col]
        if col < len(class_paths):
            with Image.open(rng.choice(class_paths)) as image:
                ax.imshow(image.convert("RGB"))
        ax.axis("off")
        if col == 0:
            ax.set_title(class_name, fontsize=9, loc="left")

fig.suptitle("Random training samples", y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# Verify the tf.data pipeline matches the serving preprocessing shape/scale.
import numpy as np

class_names = utils.list_class_dirs(train_dir)
train_ds = utils.make_dataset(train_dir, class_names, batch_size=8)

for images, labels in train_ds.take(1):
    print("batch images:", images.shape, images.dtype)
    print("batch labels:", labels.shape, labels.dtype)
    print("pixel range :", float(images.numpy().min()), "->", float(images.numpy().max()))

single = utils.preprocess_image(paths[0])
print("serving-path :", single.shape, single.dtype, f"range {single.min():.2f} -> {single.max():.2f}")
print("classes      :", class_names)

### Next steps

- Note the class names/order in `class_names.json` you intend to ship.
- Open `02_train_maize_baseline.ipynb` to train, or clone it per crop
  (`02_train_cassava_baseline.ipynb`, …).
- If imbalance is severe, compute class weights before the first training run.